# Part 1: Prepare Data

---

## Import packages

In [ ]:
import pandas as pd
from pybiomart import Dataset

## Clean and Format Data

### Count Matrix

First, read the CCLE RNA-seq counts matrix into a variable called `ccle_counts`.

In [ ]:
ccle_counts = pd.read_csv('data/CCLE_RNAseq_genes_counts_20180929.gct.gz',sep='\t', skiprows=2)

In [ ]:
print(ccle_counts.shape)
ccle_counts.head()

`ccle_counts` holds counts for 56,202 genes from 1019 cell lines (samples).

#### Deal with Duplicates

In [ ]:
# Count nubmer of unique and duplicated genes (rows).
print(f"Number of non-duplicated genes: {sum(~ccle_counts['Description'].duplicated(keep=False))}")
print(f"Number of duplicated genes: {sum(ccle_counts['Description'].duplicated(keep='first'))}")

# Count number of unique and duplicated cell lines (columns).
# num_uniq_cols = ccle_counts.columns.nunique()
print(f"Number of non-duplicated columns: {sum(~ccle_counts.columns.duplicated(keep=False))}")
print(f"Number of duplicated columns: {sum(ccle_counts.columns.duplicated(keep=False))}")

There are 1931 genes with more than 1 instance in the counts dataframe. The duplicated genes will be collapsed into single genes by summing the corresponding gene counts across duplicated rows.

In [ ]:
# Extract unique rows
ccle_counts_no_dups = ccle_counts[~ccle_counts['Description'].duplicated(keep=False)]
assert ccle_counts_no_dups.shape[0] == 54074, 'Wrong extraction'

# Extract and collapse duplicated rows
ccle_counts_dups = ccle_counts[ccle_counts['Description'].duplicated(keep='first')]
assert ccle_counts_dups.shape[0] == 1931, 'Wrong extraction'
ccle_counts_dups_summed = ccle_counts_dups.groupby('Description', group_keys=True).sum(numeric_only=True).reset_index()
print(f"Number of collapsed duplicated genes: {ccle_counts_dups_summed.shape[0]}")

We then combined non-duplicates with collapsed duplicates to form final clean count data frame:

In [ ]:
ccle_counts_cleaned = pd.concat([ccle_counts_no_dups, ccle_counts_dups_summed], axis='index', ignore_index=True)
print(f"Shape of cleaned counts dataframe: {ccle_counts_cleaned.shape}")
ccle_counts_cleaned.head()

To have an all numeric count data table, we will set genes `Description` as the dataframe's indices and drop the `Name` column:

In [ ]:
ccle_counts_cleaned = ccle_counts_cleaned.drop(columns=['Name']).set_index('Description')
ccle_counts_cleaned.head()

#### Select Protein-Coding Genes

This analysis focuses exclusively on protein-coding genes. To isolate these features, we first annotated the raw transcriptomic data and subsequently filtered the count matrix to exclude non-coding biotypes.

In [ ]:
# Initialize a connection to `hsapiens_gene_ensembl` genomic database within the BioMart system
dataset = Dataset(name='hsapiens_gene_ensembl', host='http://www.ensembl.org')

# Fetch protein-coding biotypes from the database
coding_genes_hgnc = dataset.query(
    attributes=['ensembl_gene_id', 'hgnc_symbol', 'gene_biotype'],
    filters={'biotype': ['protein_coding']}
)

# BioMart hint
print(
    """We can use `dataset.list_filters() and dataset.list_attributes()` to 
    view available filters abd attributes in BioMart system
""")

coding_genes_hgnc

In [ ]:
# Retain rows carrying count data for protein-coding genes
ccle_counts_cleaned = ccle_counts_cleaned[ccle_counts_cleaned.index.isin(coding_genes_hgnc['HGNC symbol'])]
print(f"Retained {ccle_counts_cleaned.shape[0]} protein-coding genes in the count table.")

### Metadata

In RNA-seq workflows, **metadata** refers to sample-level information—such as tissue origin, disease state, treatment conditions, or mutations—that describes the samples but is not part of the expression counts themselves. We need to ensure that the **Sample IDs** in the metadata perfectly match the **Column Names** of the counts matrix to allow for accurate data merging during analysis.

Next, we read `data/Cell_lines_annotations_20181226.txt` into `ccle_meta` dataframe:

In [ ]:
ccle_meta = pd.read_csv('data/Cell_lines_annotations_20181226.txt', sep='\t')
print(f"There are {ccle_meta.shape[0]} rows representing cell lines and "
    f"{ccle_meta.shape[1]} columns representing various information about cell lines.")
ccle_meta

To make our `ccle_meta` table leaner, we go through information under each column and determine which columns we need to preserve for downstream analysis. We also remove columns that have too many NA's.

In [ ]:
# List important columns
meta_cols_to_keep = ["CCLE_ID","depMapID","Name","Pathology","Site_Primary","Gender","tcga_code"]
ccle_meta_cleaned = ccle_meta[meta_cols_to_keep]
ccle_meta_cleaned

To maintain data integrity, we are replacing all null and empty cells with a 'missing_info' placeholder. Standardizing these gaps now ensures that missing values are handled consistently during downstream analysis and can be targeted for future inference:

In [ ]:
# Print out fraction of NA values in each column
for col in meta_cols_to_keep:
    na_fraction = sum(ccle_meta_cleaned[col].isna()) / ccle_meta_cleaned.shape[0] * 100
    print(f"{na_fraction} of {col} is missing value")

Based on data above, missing values in the following columns should be replaced with `missing_info`:
`Name`, `Pathology`, `Site_Primary`, `Gender`, and `tcga_code`

In [ ]:
ccle_meta_cleaned = ccle_meta_cleaned.fillna('missing_info')
ccle_meta_cleaned['Gender'].value_counts(dropna=False)

### Align Metadata with Counts Matrix

While each row in `ccle_meta_cleaned` represents one cell line, it is not a one-to-one match to each column in `ccle_counts_cleaned`. So we need to do additional work to align the rows of `ccle_meta_cleaned` to the columns of `ccle_counts_cleaned`.

In [ ]:
ccle_meta_cleaned_aligned = ccle_meta_cleaned[ccle_meta_cleaned['CCLE_ID'].isin(ccle_counts_cleaned.columns.values)].reset_index(drop=True)
ccle_counts_cleaned_aligned = ccle_counts_cleaned[ccle_meta_cleaned_aligned['CCLE_ID'].values]

print(f"Shape of `ccle_counts_cleaned_aligned` = {ccle_counts_cleaned_aligned.shape}")
print(f"Shape of `ccle_meta_cleaned_aligned` = {ccle_meta_cleaned_aligned.shape}")

There are **1019 cell lines** left in both counts and meta tables.

## Subset Lung Cancer Cell Lines

Because the CCLE RNA-seq dataset can be quite large to work with, we will focus on a smaller subset. We will select the cell lines derived from lung cancer because that is the indication-specific subset with the greatest number of samples in CCLE.

We can use `Site_Primary` to select **lung cancer** cell lines. We also want to remove cell lines that have missing info on `Pathology` column of the meta table:

In [ ]:
print(ccle_meta_cleaned_aligned['Site_Primary'].unique())
print(ccle_meta_cleaned_aligned['Pathology'].value_counts())

In [ ]:
ccle_meta_subset = ccle_meta_cleaned_aligned[ccle_meta_cleaned_aligned['Site_Primary'] == 'lung']
ccle_meta_subset = ccle_meta_subset[ccle_meta_subset['Pathology'] != 'missing_info']
ccle_counts_subset = ccle_counts_cleaned_aligned[ccle_meta_subset['CCLE_ID'].values]

print(f"Shape of final `ccle_counts_subset` = {ccle_counts_subset.shape}")
print(f"Shape of final `ccle_meta_subset` = {ccle_meta_subset.shape}")

## Save Outputs

In [ ]:
# Save processed tables to csv file
ccle_counts_cleaned_aligned.to_csv('outs/1_ccle_counts.csv')
ccle_meta_cleaned_aligned.to_csv('outs/1_ccle_meta.csv', index=False)

# Save subset processed tables to csv file
ccle_counts_subset.to_csv('outs/1_ccle_counts_subset.csv')
ccle_meta_subset.to_csv('outs/1_ccle_meta_subset.csv', index=False)

---

In [ ]:
import mygene

mg = mygene.MyGeneInfo()
results = mg.querymany(['ENSG00000136352', 'ENSG00000105323'], 
                        scopes='ensembl.gene', 
                        fields='symbol,name,go', 
                        species='human')
results